In [1]:
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [2]:
datapath = '../data/2010-2011 Solar home electricity data.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_01"

shape: (269_735, 53)
┌──────────┬───────────┬──────────┬──────────────────────┬───┬───────┬───────┬───────┬───────┐
│ Customer ┆ Generator ┆ Postcode ┆ Consumption Category ┆ … ┆ 22:30 ┆ 23:00 ┆ 23:30 ┆ 0:00  │
│ ---      ┆ Capacity  ┆ ---      ┆ ---                  ┆   ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i64      ┆ ---       ┆ i64      ┆ str                  ┆   ┆ f64   ┆ f64   ┆ f64   ┆ f64   │
│          ┆ f64       ┆          ┆                      ┆   ┆       ┆       ┆       ┆       │
╞══════════╪═══════════╪══════════╪══════════════════════╪═══╪═══════╪═══════╪═══════╪═══════╡
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.378 ┆ 0.128 ┆ 0.078 ┆ 0.125 │
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ 1.075 │
│ 1        ┆ 3.78      ┆ 2076     ┆ GG                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ 0.0   │
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.402 ┆ 0.142 ┆ 0.12  ┆ 0.111 │
│ 1        ┆ 3.78      ┆ 2076

In [ ]:
datapath = '../data/2011-2012 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_02"

In [ ]:
datapath = '../data/2012-2013 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_03"

In [3]:
# we can get the training and testing customers from the csv file
training_customers = np.loadtxt('../data/training_customers.csv', dtype=int)
testing_customers = np.loadtxt('../data/testing_customers.csv', dtype=int)

In [ ]:
# alternatively, get all the unique customers as their own dataframes
customers = df['Customer'].unique()
# pick 80% of the random customers as training data
training_customers = np.random.choice(customers, int(0.8*len(customers)), replace=False)
# the rest of the customers are testing data
testing_customers = np.setdiff1d(customers, training_customers)

In [ ]:
# save the customers number to a csv file
np.savetxt('../data/training_customers.csv', training_customers, fmt='%s')
np.savetxt('../data/testing_customers.csv', testing_customers, fmt='%s')

In [4]:
from helper import transform_polars_df
# loop through each customer and use transform_polars_df to get the dataframe and store it in a list call dataset
training_dataset = []
for customer in training_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as training dataset: {customer}")
        print(e)
        break
    training_dataset.append(newcustomerdf)

testing_dataset = []
for customer in testing_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as testing dataset: {customer}")
        print(e)
        break
    testing_dataset.append(newcustomerdf)

In [ ]:
# provide std and mean on the different columns of the training dataset
testdf = testing_dataset[25]
# drop timestamp and time columns
testdf = testdf.drop(['Timestamp', 'Time'])
print(testdf.describe())

In [5]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env

testing_env_fns = [make_env(ds) for ds in testing_dataset]

num_step = None # pick the number of step for the simulation/none for full length
test_envs = [env_fn(num_step) for env_fn in testing_env_fns]

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [ ]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env
# Create a list of environment creation functions to build a vectorized environment.
training_env_fns = [make_env(ds) for ds in training_dataset]
#training_vec_env = DummyVecEnv(training_env_fns)
num_step = None # pick the number of step for the simulation/none for full length
train_envs = [env_fn(num_step) for env_fn in training_env_fns]

In [ ]:
# combine the test_envs and train_envs into a single list
combined_envs = test_envs + train_envs

In [6]:
selected_list = test_envs
if selected_list is test_envs:
    env_type = "test"
    env_fns = testing_env_fns
elif selected_list is train_envs:
    env_type = "train"
    env_fns = training_env_fns
else:
    env_type = "combined"
    env_fns = testing_env_fns + training_env_fns

In [16]:
from decision import Agent, run_episodes_parallel
rule_agent_kwargs = {
    'algorithm': 'rule'
}

# run episodes in the list in parallel using the rule-based agent on the training environments
episode_logs, incident_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=rule_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

[INFO] Starting 60 episodes with max_workers=12


Episodes: 100%|██████████| 60/60 [00:22<00:00,  2.65it/s]


[START] Episode 7
Sim Complete
[DONE]  Episode 7 (Elapsed: 4.37 sec)
[START] Episode 16
Sim Complete
[DONE]  Episode 16 (Elapsed: 3.98 sec)
[START] Episode 28
Sim Complete
[DONE]  Episode 28 (Elapsed: 4.02 sec)
[START] Episode 42
Sim Complete
[DONE]  Episode 42 (Elapsed: 4.09 sec)
[START] Episode 2
Sim Complete
[DONE]  Episode 2 (Elapsed: 4.22 sec)
[START] Episode 14
Sim Complete
[DONE]  Episode 14 (Elapsed: 4.16 sec)
[START] Episode 29
Sim Complete
[DONE]  Episode 29 (Elapsed: 4.18 sec)
[START] Episode 45
Sim Complete
[DONE]  Episode 45 (Elapsed: 4.07 sec)
[START] Episode 1
Sim Complete
[DONE]  Episode 1 (Elapsed: 4.42 sec)
[START] Episode 21
Sim Complete
[DONE]  Episode 21 (Elapsed: 3.99 sec)
[START] Episode 31
Sim Complete
[DONE]  Episode 31 (Elapsed: 4.04 sec)
[START] Episode 44
Sim Complete
[DONE]  Episode 44 (Elapsed: 0.05 sec)
[START] Episode 46
Sim Complete
[DONE]  Episode 46 (Elapsed: 0.00 sec)
[START] Episode 47
Sim Complete
[DONE]  Episode 47 (Elapsed: 4.12 sec)
[START] Epis

In [17]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_logs)]
rule_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/rule_{env_type}_{episode_num}_logs.parquet"
rule_all_logs.write_parquet(file_name)



In [18]:
# Incident logs
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(incident_logs)
    if df.height > 0
]
try:
    rule_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/rule_{env_type}_{episode_num}_incident_logs.parquet"
    rule_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")

[INFO] No incident logs to save.


In [ ]:
# check for schema mismatches if there are any
# Collect all schemas
schemas = [df.schema for df in dfs_with_id]

# Find the most common schema (assume it's the correct one)
from collections import Counter
schema_counts = Counter([tuple(sorted(s.items())) for s in schemas])
most_common_schema = dict(schema_counts.most_common(1)[0][0])

# Print out indices and details of DataFrames with mismatched schemas
for i, schema in enumerate(schemas):
    if dict(sorted(schema.items())) != most_common_schema:
        print(f"DF {i} schema mismatch:")
        print("Schema:", schema)
        print("Difference:", set(schema.items()) ^ set(most_common_schema.items()))

In [7]:
from decision import Agent, run_episodes_parallel, run_single
# Initialize environments and SDP agent parameters
sdp_agent_kwargs = {
    "algorithm": "sdp",
    "horizon": 48,                  # planning horizon (steps)
    "soc_resolution": 20,           # SoC discretization
    "action_resolution": 41,        # discrete actions (best ≈ 2*soc_resolution + 1)
    "use_monte_carlo": True,
    "mc_samples": 200,
    "mc_seed": None,
}

# Run a single episode for timing test
#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=sdp_agent_kwargs, render=False, display_progress=True)


# Run all episodes in parallel
sdp_episode_logs, sdp_incident_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=sdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

[INFO] Starting 60 episodes with max_workers=12


Episodes: 100%|██████████| 60/60 [2:40:53<00:00, 160.90s/it]  


[START] Episode 4
Sim Complete
[DONE]  Episode 4 (Elapsed: 1999.73 sec)
[START] Episode 20
Sim Complete
[DONE]  Episode 20 (Elapsed: 2014.31 sec)
[START] Episode 31
Sim Complete
[DONE]  Episode 31 (Elapsed: 2027.74 sec)
[START] Episode 45
Sim Complete
[DONE]  Episode 45 (Elapsed: 1974.23 sec)
[START] Episode 6
Sim Complete
[DONE]  Episode 6 (Elapsed: 1970.89 sec)
[START] Episode 16
Sim Complete
[DONE]  Episode 16 (Elapsed: 2018.62 sec)
[START] Episode 30
Sim Complete
[DONE]  Episode 30 (Elapsed: 2049.80 sec)
[START] Episode 44
Sim Complete
[DONE]  Episode 44 (Elapsed: 16.36 sec)
[START] Episode 46
Sim Complete
[DONE]  Episode 46 (Elapsed: 5.92 sec)
[START] Episode 47
Sim Complete
[DONE]  Episode 47 (Elapsed: 2024.67 sec)
[START] Episode 11
Sim Complete
[DONE]  Episode 11 (Elapsed: 1995.96 sec)
[START] Episode 18
Sim Complete
[DONE]  Episode 18 (Elapsed: 2033.04 sec)
[START] Episode 34
Sim Complete
[DONE]  Episode 34 (Elapsed: 2076.23 sec)
[START] Episode 49
Sim Complete
[DONE]  Episode

In [8]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(sdp_episode_logs)]
sdp_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/sdp_{env_type}_{episode_num}_logs.parquet"
sdp_all_logs.write_parquet(file_name)

In [9]:
# incident logs
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(sdp_incident_logs)
    if df.height > 0
]
try:
    sdp_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/sdp_{env_type}_{episode_num}_incident_logs.parquet"
    sdp_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")


[INFO] No incident logs to save.


In [10]:
from decision import Agent, run_episodes_parallel, run_single
mrdp_agent_kwargs = {
    'algorithm': 'mrdp',
    'soc_resolution': 20,           # fallback/default for single-horizon
    'action_resolution': 41,        # fallback/default for single-horizon
    'subhorizon_specs': [
        {
            'start': 0,
            'length': 12,           # e.g. 6 hours at 30-min steps
            'soc_resolution': 20,   # fine SoC discretization
            'action_resolution': 41,# fine action discretization
            'step_duration': 0.5    # hours per step (30 min)
        },
        {
            'start': 12,
            'length': 72,           # e.g. 36 hours at 30-min steps
            'soc_resolution': 8,    # coarse SoC discretization
            'action_resolution': 17, # coarse action discretization
            'step_duration': 0.5    # hours per step (30 min)
        }
    ],
    'use_monte_carlo': True,
    'mc_samples': 200,
    'mc_seed': None,
}

#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=mrdp_agent_kwargs, render=False, display_progress=True)

# Run all episodes in parallel using MRDP

mrdp_episode_logs, mrdp_incident_logs = run_episodes_parallel(
    Agent, selected_list, agent_kwargs=mrdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False
)

[INFO] Starting 60 episodes with max_workers=12


Episodes: 100%|██████████| 60/60 [2:34:35<00:00, 154.59s/it]  


[START] Episode 3
Sim Complete
[DONE]  Episode 3 (Elapsed: 1897.92 sec)
[START] Episode 16
Sim Complete
[DONE]  Episode 16 (Elapsed: 1904.98 sec)
[START] Episode 31
Sim Complete
[DONE]  Episode 31 (Elapsed: 1948.76 sec)
[START] Episode 44
Sim Complete
[DONE]  Episode 44 (Elapsed: 16.20 sec)
[START] Episode 46
Sim Complete
[DONE]  Episode 46 (Elapsed: 5.68 sec)
[START] Episode 47
Sim Complete
[DONE]  Episode 47 (Elapsed: 1925.03 sec)
[START] Episode 9
Sim Complete
[DONE]  Episode 9 (Elapsed: 1916.90 sec)
[START] Episode 19
Sim Complete
[DONE]  Episode 19 (Elapsed: 1919.22 sec)
[START] Episode 34
Sim Complete
[DONE]  Episode 34 (Elapsed: 1966.98 sec)
[START] Episode 49
Sim Complete
[DONE]  Episode 49 (Elapsed: 1912.53 sec)
[START] Episode 8
Sim Complete
[DONE]  Episode 8 (Elapsed: 38.45 sec)
[START] Episode 12
Sim Complete
[DONE]  Episode 12 (Elapsed: 0.11 sec)
[START] Episode 13
Sim Complete
[DONE]  Episode 13 (Elapsed: 1896.64 sec)
[START] Episode 22
Sim Complete
[DONE]  Episode 22 (El

In [11]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(mrdp_episode_logs)]
mrdp_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/mrdp_{env_type}_{episode_num}_logs.parquet"
mrdp_episode_logs.write_parquet(file_name)



In [12]:
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(mrdp_incident_logs)
    if df.height > 0
]
try:
    mrdp_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/mrdp_{env_type}_{episode_num}_incident_logs.parquet"
    mrdp_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")


[INFO] No incident logs to save.


In [13]:
from decision import Agent, run_episodes_parallel

oracle_agent_kwargs = {
    "algorithm": "oracle",
    "horizon": 48,              # 48 steps (~24 h @ 0.5h/step)
    "soc_resolution": 20,       # SoC discretization levels
    "action_resolution": 41    # ≈ 2*soc_resolution + 1 (fine enough)
}

# Run all episodes in parallel using the oracle agent
oracle_episode_logs, oracle_incident_logs = run_episodes_parallel(
    Agent,
    selected_list,  # your list of environments
    agent_kwargs=oracle_agent_kwargs,
    max_workers=14,
    use_notebook_tqdm=False
)

[INFO] Starting 60 episodes with max_workers=14


Episodes: 100%|██████████| 60/60 [14:21:51<00:00, 861.86s/it]   


[START] Episode 7
Sim Complete
[DONE]  Episode 7 (Elapsed: 11236.35 sec)
[START] Episode 20
Sim Complete
[DONE]  Episode 20 (Elapsed: 11273.54 sec)
[START] Episode 35
Sim Complete
[DONE]  Episode 35 (Elapsed: 10311.42 sec)
[START] Episode 47
Sim Complete
[DONE]  Episode 47 (Elapsed: 10951.24 sec)
[START] Episode 13
Sim Complete
[DONE]  Episode 13 (Elapsed: 10524.89 sec)
[START] Episode 15
Sim Complete
[DONE]  Episode 15 (Elapsed: 11194.17 sec)
[START] Episode 33
Sim Complete
[DONE]  Episode 33 (Elapsed: 10983.58 sec)
[START] Episode 45
Sim Complete
[DONE]  Episode 45 (Elapsed: 12127.10 sec)
[START] Episode 1
Sim Complete
[DONE]  Episode 1 (Elapsed: 11193.46 sec)
[START] Episode 18
Sim Complete
[DONE]  Episode 18 (Elapsed: 10435.86 sec)
[START] Episode 32
Sim Complete
[DONE]  Episode 32 (Elapsed: 11145.10 sec)
[START] Episode 46
Sim Complete
[DONE]  Episode 46 (Elapsed: 1861.95 sec)
[START] Episode 50
Sim Complete
[DONE]  Episode 50 (Elapsed: 10950.43 sec)
[START] Episode 6
Sim Complete

In [14]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(oracle_episode_logs)]
oracle_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/oracle_{env_type}_{episode_num}_logs.parquet"
oracle_episode_logs.write_parquet(file_name)

In [15]:
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(oracle_incident_logs)
    if df.height > 0
]
try:
    oracle_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/oracle_{env_type}_{episode_num}_incident_logs.parquet"
    oracle_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")

[INFO] No incident logs to save.


In [ ]:
from decision import Agent, run_sb3_model_on_vec_env
from stable_baselines3 import PPO, A2C, DDPG, SAC, TD3
from helper import flatten_episode_data

# (only needed if you ever switch to SubprocVecEnv on Linux/notebooks)
multiprocessing.set_start_method("forkserver", force=True)

# Utility to yield batches from a list
def batchify(lst, batch_size):
    """Yield successive batches from lst of size batch_size."""
    for i in range(0, len(lst), batch_size):
        yield lst[i:i + batch_size]

from stable_baselines3.common.vec_env import SubprocVecEnv

batch_size = 64  # Set your desired batch size

In [ ]:
sac_model = SAC.load("../models/sac_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(sac_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

sac_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/sac_{env_type}_{episode_num}_logs.parquet"
sac_logs.write_parquet(file_name)

In [ ]:
PPO_model = PPO.load("../models/ppo_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(PPO_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

ppo_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ppo_{env_type}_{episode_num}_logs.parquet"
ppo_logs.write_parquet(file_name)

In [ ]:
a2c_model = A2C.load("../models/a2c_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(a2c_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

a2c_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/a2c_{env_type}_{episode_num}_logs.parquet"
a2c_logs.write_parquet(file_name)

In [ ]:
ddpg_model = DDPG.load("../models/ddpg_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(ddpg_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
ddpg_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ddpg_{env_type}_{episode_num}_logs.parquet"
ddpg_logs.write_parquet(file_name)

In [ ]:
td3_model = TD3.load("../models/td3_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(td3_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
td3_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/td3_{env_type}_{episode_num}_logs.parquet"
td3_logs.write_parquet(file_name)

In [ ]:
# to get the best RTG value, analysis of the distribution of total episode rewards in the dataset is needed


In [ ]:
from decision import Agent, run_episodes_parallel, run_single
from decision_transformer import DecisionTransformer
import json

import torch
# check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

with open('../models/decision_transformer_model_kwargs.json', 'r') as f:
    model_kwargs = json.load(f)

model = DecisionTransformer(**model_kwargs)

model.load_state_dict(torch.load('../models/dt_model.pt', map_location=device))
model.return_scale = 1.0  # or whatever was used during training
model.eval()
rtg = 4508964.69


dt_agent_kwargs = {
    'algorithm': 'dt',
    'model': model.to(device),
    'rtg_value': rtg
}

# check for nan in model parameters
import math
bad_params = [name for name, p in model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any()]
print("bad params:", bad_params)

In [ ]:
episode_log = run_episodes_parallel(Agent, test_envs, agent_kwargs=dt_agent_kwargs, max_workers=2, use_notebook_tqdm=False)

dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_log)]
dt_logs = pl.concat(dfs_with_id)
file_name = f"../data/dt_rtg{int(rtg)}_{env_type}_{episode_num}_logs.parquet"
dt_logs.write_parquet(file_name)